# Gradient Analysis

**Table 1 — LTI (single event):** CGA and FGA for all parameters, TD vs FS, MRSTFT vs SOT

**Table 2 — LTV (multi-event):** Gradient Assignment Accuracy for onset time,
measuring *any-target* (per-event), and *matched-target* (joint).

In [47]:
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.optimize import linear_sum_assignment
from itertools import product as iterproduct

from src.synths.synth import Synth, SynthConfig
from src.synths.ddsp import Implementation
from src.losses import MultiScaleSpectralLoss, SOT2048Loss

In [50]:
# ═══════════════════════════════════════════════════════════════════
#                         CONFIGURATION
# ═══════════════════════════════════════════════════════════════════

FS = 16000
NUM_SAMPLES = FS * 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_INIT = 250
BATCH_SIZE = 32
FGA_FRACTION = 0.10   # ±10 % of full range around target

print(f"Device: {DEVICE}")

# Ground-truth event parameters (shared across both tables)
GT_EVENT = {
    "f0":             110.0,
    "time":           0.5,
    "a1":             0.2,
    "decay":          0.99,
    "pluck_position": 0.25,
    "burst_gain":     0.9,
    "dynamic_level":  0.9,
}

# CGA sweep ranges (full valid domain per parameter)
PARAM_RANGES = {
    "f0":             (55.0,  880.0),
    "decay":          (0.9,  0.999),
    "a1":             (0.001,  0.5),
    "pluck_position": (0.01,  0.49),
    "dynamic_level":  (0.5,  1.0),
    "burst_gain":     (0.5,  1),
    "time":           (0.01,  0.99),
}

PARAM_ORDER = [
    "f0", "decay", "a1", "pluck_position",
    "dynamic_level", "burst_gain", "time",
]

# Multi-event target placement: t_k = k/K − 1/(2K)
def event_times(K):
    return [k / K - 1 / (2 * K) for k in range(1, K + 1)]

EVENT_COUNTS = [2, 3, 4, 5]

# Print target placements for reference
for K in EVENT_COUNTS:
    print(f"  K={K}: {[f'{t:.3f}' for t in event_times(K)]}")

Device: cpu
  K=2: ['0.250', '0.750']
  K=3: ['0.167', '0.500', '0.833']
  K=4: ['0.125', '0.375', '0.625', '0.875']
  K=5: ['0.100', '0.300', '0.500', '0.700', '0.900']


In [51]:
# ═══════════════════════════════════════════════════════════════════
#                       BUILD SYNTHESIZERS
# ═══════════════════════════════════════════════════════════════════

synth_td = Synth(SynthConfig(
    num_samples=NUM_SAMPLES, fs=FS,
    implementation=Implementation.TIME_DOMAIN,
)).to(DEVICE)

synth_fs_lti = Synth(SynthConfig(
    num_samples=NUM_SAMPLES, fs=FS,
    implementation=Implementation.FREQUENCY_SAMPLING,
    use_lti=True,
)).to(DEVICE)

synth_fs_ltv = Synth(SynthConfig(
    num_samples=NUM_SAMPLES, fs=FS,
    implementation=Implementation.FREQUENCY_SAMPLING,
    n_fft=16384,
    hop_length=256,
    use_lti=False,
)).to(DEVICE)

mss_fn = MultiScaleSpectralLoss().to(DEVICE)
sot_fn = SOT2048Loss(sample_rate=FS).to(DEVICE)

SYNTHS_LTI  = {"TD": synth_td, "FS": synth_fs_lti}
SYNTHS_LTV  = {"TD": synth_td, "FS": synth_fs_ltv}
LOSSES      = {"MRSTFT": mss_fn, "SOT": sot_fn}

print("Synthesizers and losses ready.")

Synthesizers and losses ready.


In [52]:
# ═══════════════════════════════════════════════════════════════════
#                        SHARED HELPERS
# ═══════════════════════════════════════════════════════════════════

def make_params(K, times, device=DEVICE, batch_size=None):
    """
    Build a full parameter dict for K events.

    Parameters
    ----------
    K : int
        Number of events.
    times : list[float] | Tensor[B, K]
        Event onset times.
    batch_size : int | None
        If given and `times` is a list, broadcast to this batch size.
    """
    if isinstance(times, (list, tuple)):
        times = torch.tensor([times], dtype=torch.float32, device=device)
        if batch_size is not None and batch_size > 1:
            times = times.expand(batch_size, -1).clone()

    B = times.shape[0]
    return {
        "exists":         torch.ones(B, K, device=device),
        "time":           times,
        "f0":             torch.full((B, K), GT_EVENT["f0"],             device=device),
        "burst_gain":     torch.full((B, K), GT_EVENT["burst_gain"],     device=device),
        "pluck_position": torch.full((B, K), GT_EVENT["pluck_position"], device=device),
        "dynamic_level":  torch.full((B, K), GT_EVENT["dynamic_level"],  device=device),
        "a1":             torch.full((B, K), GT_EVENT["a1"],             device=device),
        "decay":          torch.full((B, K), GT_EVENT["decay"],          device=device),
    }

---
## Table 1 —  (single event, all parameters)

In [53]:
# ═══════════════════════════════════════════════════════════════════
#                 TABLE 1: GENERATE TARGET AUDIO
# ═══════════════════════════════════════════════════════════════════

with torch.no_grad():
    gt_params_lti = make_params(1, [GT_EVENT["time"]])
    target_audio_lti, _ = synth_td.oracle_synth(gt_params_lti)

print(f"LTI target — shape: {target_audio_lti.shape}, "
      f"peak: {target_audio_lti.abs().max():.4f}")

LTI target — shape: torch.Size([1, 64000]), peak: 0.6279


In [54]:
# ═══════════════════════════════════════════════════════════════════
#              TABLE 1: BATCHED GRADIENT COMPUTATION
# ═══════════════════════════════════════════════════════════════════

def compute_gradients_batched(
    synth, loss_fn, base_params, sweep_key, sweep_values, target_audio,
    batch_size=BATCH_SIZE,
):
    """
    Return gradient of `loss_fn` w.r.t. `sweep_key` at each value in
    `sweep_values`.  Processes in mini-batches for speed.

    The loss is averaged over batch items internally, but because each
    batch item's audio depends only on its own parameter value, gradient
    *signs* are identical to the per-sample case.
    """
    N = len(sweep_values)
    all_grads = np.empty(N)

    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        B = end - start

        params = {
            k: v.expand(B, -1).clone().detach()
            for k, v in base_params.items()
        }

        sweep_t = torch.tensor(
            sweep_values[start:end],
            dtype=torch.float32, device=DEVICE,
        ).unsqueeze(1)
        sweep_t.requires_grad_(True)
        params[sweep_key] = sweep_t

        pred_audio, _ = synth(params)
        loss = loss_fn(pred_audio, target_audio.expand(B, -1))
        loss.backward()

        all_grads[start:end] = sweep_t.grad[:, 0].detach().cpu().numpy()

    return all_grads


def gradient_accuracy(grads, sweep_values, target_val):
    """
    Fraction of initialisations where gradient descent would move the
    parameter toward the target.

    Correct when  sign(−∇) == sign(target − pred),
    equivalently    ∇ · (target − pred) < 0.
    """
    delta = target_val - sweep_values
    mask = np.abs(delta) > 1e-8           # skip points already at target
    if mask.sum() == 0:
        return np.nan
    correct = (grads[mask] * delta[mask]) < 0
    return float(correct.mean())

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                      TABLE 1: RUN SWEEPS
# ═══════════════════════════════════════════════════════════════════

table1 = {}
base_params_lti = make_params(1, [GT_EVENT["time"]])

for param_name in tqdm(PARAM_ORDER, desc="Table 1 · params"):
    lo, hi = PARAM_RANGES[param_name]
    target_val = GT_EVENT[param_name]
    span = hi - lo

    cga_vals = np.linspace(lo, hi, N_INIT)

    fga_lo = max(lo, target_val - FGA_FRACTION * span)
    fga_hi = min(hi, target_val + FGA_FRACTION * span)
    fga_vals = np.linspace(fga_lo, fga_hi, N_INIT)

    for impl_name, synth in SYNTHS_LTI.items():
        for loss_name, loss_fn in LOSSES.items():

            g_cga = compute_gradients_batched(
                synth, loss_fn, base_params_lti,
                param_name, cga_vals, target_audio_lti,
            )
            g_fga = compute_gradients_batched(
                synth, loss_fn, base_params_lti,
                param_name, fga_vals, target_audio_lti,
            )

            cga = gradient_accuracy(g_cga, cga_vals, target_val)
            fga = gradient_accuracy(g_fga, fga_vals, target_val)

            table1[(param_name, impl_name, loss_name)] = {
                "CGA": cga, "FGA": fga,
            }

            tqdm.write(
                f"  {param_name:16s} | {impl_name} | {loss_name:6s} "
                f"| CGA={cga:.1%}  FGA={fga:.1%}"
            )

Table 1 · params:   0%|          | 0/7 [00:00<?, ?it/s]

  f0               | TD | MRSTFT | CGA=68.0%  FGA=58.4%
  f0               | TD | SOT    | CGA=54.8%  FGA=50.8%
  f0               | FS | MRSTFT | CGA=69.6%  FGA=63.6%
  f0               | FS | SOT    | CGA=50.0%  FGA=50.0%
  decay            | TD | MRSTFT | CGA=100.0%  FGA=100.0%
  decay            | TD | SOT    | CGA=70.4%  FGA=47.6%
  decay            | FS | MRSTFT | CGA=100.0%  FGA=99.6%
  decay            | FS | SOT    | CGA=73.6%  FGA=47.6%
  a1               | TD | MRSTFT | CGA=100.0%  FGA=100.0%
  a1               | TD | SOT    | CGA=40.0%  FGA=50.0%
  a1               | FS | MRSTFT | CGA=99.2%  FGA=94.8%
  a1               | FS | SOT    | CGA=34.4%  FGA=50.0%
  pluck_position   | TD | MRSTFT | CGA=66.8%  FGA=89.2%
  pluck_position   | TD | SOT    | CGA=49.6%  FGA=48.0%


In [30]:
# ═══════════════════════════════════════════════════════════════════
#                    TABLE 1: FORMAT & DISPLAY
# ═══════════════════════════════════════════════════════════════════

PARAM_LABELS = {
    "f0":             r"$f_0$",
    "decay":          r"$g$ (decay)",
    "a1":             r"$a_1$ (damping)",
    "pluck_position": r"Pluck position",
    "dynamic_level":  r"Dynamic level",
    "burst_gain":     r"Burst gain",
    "time":           r"Onset $t$",
}

rows1 = []
for p in PARAM_ORDER:
    row = {"Parameter": PARAM_LABELS[p]}
    for loss_name in ["MRSTFT", "SOT"]:
        for impl in ["TD", "FS"]:
            for metric in ["CGA", "FGA"]:
                col = f"{loss_name} · {impl} · {metric}"
                val = table1[(p, impl, loss_name)][metric]
                row[col] = f"{val:.1%}" if not np.isnan(val) else "—"
    rows1.append(row)

df1 = pd.DataFrame(rows1).set_index("Parameter")

cols1 = [
    f"{L} · {I} · {M}"
    for L in ["MRSTFT", "SOT"]
    for I in ["TD", "FS"]
    for M in ["CGA", "FGA"]
]
df1 = df1[cols1]

print(df1.to_string())

df1.to_csv("gradient_accuracy_single.csv")


╔══════════════════════════════════════════════════════════╗
║     Table 1 — LTI Gradient Reliability (1 event)       ║
╚══════════════════════════════════════════════════════════╝

                MRSTFT · TD · CGA MRSTFT · TD · FGA MRSTFT · FS · CGA MRSTFT · FS · FGA SOT · TD · CGA SOT · TD · FGA SOT · FS · CGA SOT · FS · FGA
Parameter                                                                                                                                          
$f_0$                       68.0%             58.4%             69.6%             63.6%          54.8%          50.8%          50.0%          50.0%
$g$ (decay)                100.0%            100.0%            100.0%             99.6%          70.4%          47.6%          73.6%          47.6%
$a_1$ (damping)            100.0%            100.0%             99.2%             94.8%          40.0%          50.0%          34.4%          50.0%
Pluck position              66.8%             89.2%             67.6%        

---
## Table 2 — Multi-Event Gradient Analysis
| Metric    | Definition |
|:----------|:-----------|
| **Any**   | Is each event's gradient pointing toward *some* real target? |
| **Joint** | Are *all* K gradients simultaneously correct? |

In [31]:
# ═══════════════════════════════════════════════════════════════════
#              TABLE 2: GRADIENT CHECKING HELPERS
# ═══════════════════════════════════════════════════════════════════

def hungarian_match(pred_times, target_times):
    """
    Match predictions → targets by L1 distance (Hungarian algorithm).

    Returns
    -------
    assignment : ndarray of int, shape [K]
        assignment[i] = index of target matched to prediction i
    """
    cost = np.abs(pred_times[:, None] - target_times[None, :])
    row_ind, col_ind = linear_sum_assignment(cost)
    assignment = np.empty(len(pred_times), dtype=int)
    assignment[row_ind] = col_ind
    return assignment


def check_any_target(grads, pred_times, target_times, eps=1e-12):
    """
    Per-event: does −∇ point toward *any* target?
    """
    K = len(grads)
    correct = np.zeros(K, dtype=bool)
    for i in range(K):
        if np.abs(grads[i]) < eps:
            continue  # zero gradient → not useful
        if -grads[i] > 0:          # descent moves rightward
            correct[i] = np.any(target_times > pred_times[i] + eps)
        else:                       # descent moves leftward
            correct[i] = np.any(target_times < pred_times[i] - eps)
    return correct


def check_matched_target(grads, pred_times, target_times, assignment, eps=1e-8):
    """
    Per-event: does −∇ point toward the *Hungarian-matched* target?
    """
    K = len(grads)
    correct = np.zeros(K, dtype=bool)
    for i in range(K):
        direction = target_times[assignment[i]] - pred_times[i]
        if np.abs(direction) < eps:
            correct[i] = True       # already at target
            continue
        if np.abs(grads[i]) < 1e-12:
            continue                # zero gradient → not useful
        correct[i] = (grads[i] * direction) < 0  # −∇ aligns with direction
    return correct

In [32]:
# ═══════════════════════════════════════════════════════════════════
#           TABLE 2: BUILD INITIALISATION GRIDS
# ═══════════════════════════════════════════════════════════════════

def make_init_grid(K, n_target=N_INIT, lo=0.01, hi=0.99):
    """
    ~n_target equally-spaced K-dimensional combinations.
    Returns ndarray of shape [n_per_dim**K, K].
    """
    n_per_dim = max(2, int(round(n_target ** (1.0 / K))))
    axis = np.linspace(lo, hi, n_per_dim)
    grid = np.array(list(iterproduct(*([axis] * K))))
    return grid


print("Initialisation grids:")
for K in EVENT_COUNTS:
    g = make_init_grid(K)
    n_per_dim = int(round(N_INIT ** (1.0 / K)))
    print(f"  K={K}: {n_per_dim} per dim → {g.shape[0]:,} combos")

Initialisation grids:
  K=1: 250 per dim → 250 combos
  K=2: 16 per dim → 256 combos
  K=3: 6 per dim → 216 combos
  K=4: 4 per dim → 256 combos
  K=5: 3 per dim → 243 combos


In [33]:
# ═══════════════════════════════════════════════════════════════════
#                       TABLE 2: RUN
# ═══════════════════════════════════════════════════════════════════

table2 = {}

for K in tqdm(EVENT_COUNTS, desc="Table 2 · event counts"):

    # --- target audio for this K ---
    tgt_t = event_times(K)
    with torch.no_grad():
        tgt_params = make_params(K, tgt_t)
        target_audio_ltv, _ = synth_td.oracle_synth(tgt_params)

    target_times = np.array(tgt_t)
    grid = make_init_grid(K)
    n_combos = grid.shape[0]

    for impl_name, synth in SYNTHS_LTV.items():
        for loss_name, loss_fn in LOSSES.items():

            any_per_event   = []
            match_per_event = []
            match_joint     = []

            for start in tqdm(
                range(0, n_combos, BATCH_SIZE),
                desc=f"  K={K} {impl_name} {loss_name}",
                leave=False,
            ):
                end = min(start + BATCH_SIZE, n_combos)
                B = end - start
                batch_preds = grid[start:end]           # [B, K]

                time_t = torch.tensor(
                    batch_preds,
                    dtype=torch.float32, device=DEVICE,
                )
                time_t.requires_grad_(True)

                params = {
                    "exists":         torch.ones(B, K, device=DEVICE),
                    "time":           time_t,
                    "f0":             torch.full((B, K), GT_EVENT["f0"],             device=DEVICE),
                    "burst_gain":     torch.full((B, K), GT_EVENT["burst_gain"],     device=DEVICE),
                    "pluck_position": torch.full((B, K), GT_EVENT["pluck_position"], device=DEVICE),
                    "dynamic_level":  torch.full((B, K), GT_EVENT["dynamic_level"],  device=DEVICE),
                    "a1":             torch.full((B, K), GT_EVENT["a1"],             device=DEVICE),
                    "decay":          torch.full((B, K), GT_EVENT["decay"],          device=DEVICE),
                }

                pred_audio, _ = synth(params)
                loss = loss_fn(pred_audio, target_audio_ltv.expand(B, -1))
                loss.backward()

                grads_np = time_t.grad.detach().cpu().numpy()

                for b in range(B):
                    g = grads_np[b]
                    p = batch_preds[b]

                    any_c   = check_any_target(g, p, target_times)
                    assign  = hungarian_match(p, target_times)
                    match_c = check_matched_target(g, p, target_times, assign)

                    any_per_event.append(any_c.mean())
                    match_per_event.append(match_c.mean())
                    match_joint.append(match_c.all())

            table2[(K, impl_name, loss_name)] = {
                "Any (per-event)":     np.mean(any_per_event),
                "Matched (per-event)": np.mean(match_per_event),
                "Matched (joint)":     np.mean(match_joint),
            }

            tqdm.write(
                f"  K={K} {impl_name} {loss_name}: "
                f"Any={np.mean(any_per_event):.1%}  "
                f"Matched={np.mean(match_per_event):.1%}  "
                f"Joint={np.mean(match_joint):.1%}"
            )

Table 2 · event counts:   0%|          | 0/5 [00:00<?, ?it/s]

  K=1 TD MRSTFT:   0%|          | 0/8 [00:00<?, ?it/s]

  K=1 TD MRSTFT: Any=51.2%  Matched=51.2%  Joint=51.2%


  K=1 TD SOT:   0%|          | 0/8 [00:00<?, ?it/s]

  K=1 TD SOT: Any=42.4%  Matched=42.4%  Joint=42.4%


...

  K=5 TD SOT: Any=66.9%  Matched=58.4%  Joint=1.2%


  K=5 FS MRSTFT:   0%|          | 0/8 [00:00<?, ?it/s]

  K=5 FS MRSTFT: Any=65.3%  Matched=56.4%  Joint=3.3%


  K=5 FS SOT:   0%|          | 0/8 [00:00<?, ?it/s]

  K=5 FS SOT: Any=68.6%  Matched=60.5%  Joint=4.1%


In [34]:
# ═══════════════════════════════════════════════════════════════════
#                   TABLE 2: FORMAT & DISPLAY
# ═══════════════════════════════════════════════════════════════════

METRIC_ORDER = ["Any (per-event)", "Matched (per-event)", "Matched (joint)"]

rows2 = []
for K in EVENT_COUNTS:
    for metric in METRIC_ORDER:
        row = {"Events": K, "Metric": metric}
        for loss_name in ["MRSTFT", "SOT"]:
            for impl in ["TD", "FS"]:
                col = f"{loss_name} · {impl}"
                val = table2[(K, impl, loss_name)][metric]
                row[col] = f"{val:.1%}"
        rows2.append(row)

df2 = pd.DataFrame(rows2).set_index(["Events", "Metric"])

cols2 = [f"{L} · {I}" for L in ["MRSTFT", "SOT"] for I in ["TD", "FS"]]
df2 = df2[cols2]
print(df2.to_string())

df2.to_csv("gradient_accuracy_multi_event.csv")


╔══════════════════════════════════════════════════════════╗
║   Table 2 — LTV Gradient Assignment Accuracy (time)    ║
╚══════════════════════════════════════════════════════════╝

                           MRSTFT · TD MRSTFT · FS SOT · TD SOT · FS
Events Metric                                                       
1      Any (per-event)           51.2%       58.8%    42.4%    74.4%
       Matched (per-event)       51.2%       58.8%    42.4%    74.4%
       Matched (joint)           51.2%       58.8%    42.4%    74.4%
2      Any (per-event)           73.4%       77.1%    64.1%    77.5%
       Matched (per-event)       52.0%       48.6%    37.9%    50.8%
       Matched (joint)           22.7%       23.0%    12.1%    28.5%
3      Any (per-event)           83.3%       82.7%    79.9%    84.9%
       Matched (per-event)       47.4%       49.8%    42.6%    50.3%
       Matched (joint)            7.4%       13.9%     5.6%    11.6%
4      Any (per-event)           71.6%       72.8%    70.7